# Chapter 34 Walkthrough — Error Analysis, Subgroup Fairness, Calibration & Responsible Clinical AI

**Book Reference:** Chapter 34, Stage 3 & Stage 4 (`ch34_error_analysis_fairness_calibration.qmd`)  
Run every cell top-to-bottom to see why aggregate AUC hides rural and wealth-quintile disparities on `mch_survey_sample`, audit subgroup calibration (ECE & Brier), slice worst FN/FP errors, mitigate Equal Opportunity gaps via validation-calibrated group thresholds, and monitor covariate shift with the Population Stability Index (PSI).

In [ ]:
# @title Step 0: Install sharukh-tutor & Train Baseline Stunting Model
!pip install -q sharukh-tutor
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
df = mch_raw.copy()
df["residence"] = df["residence"].astype(str).str.strip().str.title()
df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
df["is_urban"] = (df["residence"] == "Urban").astype(int)
w_map = {
    "Poorest": 1, "Poorer": 2, "Middle": 3,
    "Richer": 4, "Richest": 5,
}
df["wealth_rank"] = df["wealth_quintile"].map(w_map)
feat_cols = [
    "maternal_age_years", "anc_visits", "maternal_hb_g_dl",
    "wealth_rank", "is_urban",
]
keep = [
    "household_id", "residence", "wealth_quintile", "child_stunted",
] + feat_cols
clean = df[keep].dropna().reset_index(drop=True)

X = clean[feat_cols]
y = clean["child_stunted"].astype(int)
X_tr, X_te, y_tr, y_te, df_tr, df_te = train_test_split(
    X, y, clean, test_size=0.30, stratify=y, random_state=42
)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500)),
])
pipe.fit(X_tr, y_tr)
p_tr = pipe.predict_proba(X_tr)[:, 1]
p_te = pipe.predict_proba(X_te)[:, 1]
fpr_tr, tpr_tr, th_tr = roc_curve(y_tr, p_tr)
t_youden = float(th_tr[int(np.argmax(tpr_tr - fpr_tr))])
pred_te = (p_te >= t_youden).astype(int)

print(f"Complete rows: {len(clean)} | Test rows: {len(y_te)}")
print(
    f"Overall Test ROC-AUC: {roc_auc_score(y_te, p_te):.4f} | "
    f"Train Youden t: {t_youden:.4f}"
)


## Section 34.3.1 — Aggregate AUC Hides Subgroup Disparities

In [ ]:
def subgroup_performance_table(y_true, y_prob, groups, threshold=0.5):
    """Disaggregate clinical classifier metrics by sorted subgroup."""
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_prob, dtype=float)
    gr = np.asarray(groups)
    rows = []
    for g in sorted(pd.unique(gr)):
        m = gr == g
        yt_g, yp_g = yt[m], yp[m]
        pred_g = (yp_g >= threshold).astype(int)
        auc = (
            round(float(roc_auc_score(yt_g, yp_g)), 4)
            if len(np.unique(yt_g)) == 2
            else np.nan
        )
        tp = int(np.sum((yt_g == 1) & (pred_g == 1)))
        fn = int(np.sum((yt_g == 1) & (pred_g == 0)))
        fp = int(np.sum((yt_g == 0) & (pred_g == 1)))
        tn = int(np.sum((yt_g == 0) & (pred_g == 0)))
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        rows.append({
            "group": g.item() if hasattr(g, "item") else g,
            "n": int(np.sum(m)),
            "prevalence": round(float(np.mean(yt_g == 1)), 4),
            "roc_auc": auc,
            "tpr": round(tpr, 4),
            "fpr": round(fpr, 4),
            "brier": round(float(brier_score_loss(yt_g, yp_g)), 4),
        })
    cols = [
        "group", "n", "prevalence", "roc_auc", "tpr", "fpr", "brier",
    ]
    return pd.DataFrame(rows, columns=cols)

print("--- Residence Subgroups ---")
print(
    subgroup_performance_table(
        y_te, p_te, df_te["residence"], threshold=t_youden
    ).to_string(index=False)
)
print("\n--- Wealth Quintile Subgroups ---")
print(
    subgroup_performance_table(
        y_te, p_te, df_te["wealth_quintile"], threshold=t_youden
    ).to_string(index=False)
)


## Section 34.3.2 — Quantifying Fairness Gaps: Demographic Parity, Equal Opportunity & Equalized Odds

In [ ]:
def compute_fairness_gaps(y_true, y_pred, sensitive_attr):
    """Compute max-min DP, Equal Opportunity, FPR, and EqOdds gaps."""
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_pred, dtype=int)
    gr = np.asarray(sensitive_attr)
    if not (len(yt) == len(yp) == len(gr)):
        raise ValueError(
            "y_true, y_pred, and sensitive_attr must have equal length"
        )
    sels, tprs, fprs = [], [], []
    for g in sorted(pd.unique(gr)):
        m = gr == g
        yt_g, yp_g = yt[m], yp[m]
        sels.append(float(np.mean(yp_g == 1)))
        tp = int(np.sum((yt_g == 1) & (yp_g == 1)))
        fn = int(np.sum((yt_g == 1) & (yp_g == 0)))
        fp = int(np.sum((yt_g == 0) & (yp_g == 1)))
        tn = int(np.sum((yt_g == 0) & (yp_g == 0)))
        tprs.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
    eo = max(tprs) - min(tprs)
    fpr_d = max(fprs) - min(fprs)
    return {
        "demographic_parity_diff": round(max(sels) - min(sels), 4),
        "equal_opportunity_diff": round(eo, 4),
        "max_fpr_diff": round(fpr_d, 4),
        "equalized_odds_diff": round(max(eo, fpr_d), 4),
    }

for label, col in [
    ("Residence", "residence"),
    ("Wealth", "wealth_quintile"),
]:
    gaps = compute_fairness_gaps(y_te, pred_te, df_te[col])
    print(
        f"{label:9s} | DP={gaps['demographic_parity_diff']:.4f} | "
        f"EO={gaps['equal_opportunity_diff']:.4f} | "
        f"FPR={gaps['max_fpr_diff']:.4f} | "
        f"EqOdds={gaps['equalized_odds_diff']:.4f}"
    )


## Section 34.3.3 — Subgroup Calibration Audit (ECE & Brier Disparity)

In [ ]:
def subgroup_calibration_audit(y_true, y_prob, groups, n_bins=5):
    """Audit equal-width ECE and Brier score across sorted subgroups."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_prob, dtype=float)
    gr = np.asarray(groups)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows, raw_eces = [], {}
    for g in sorted(pd.unique(gr)):
        m = gr == g
        yt_g, yp_g = yt[m], yp[m]
        n_g = int(np.sum(m))
        ece = 0.0
        for i in range(n_bins):
            lo, hi = edges[i], edges[i + 1]
            if i < n_bins - 1:
                bm = (yp_g >= lo) & (yp_g < hi)
            else:
                bm = (yp_g >= lo) & (yp_g <= hi)
            cnt = int(np.sum(bm))
            if cnt > 0:
                gap = abs(float(yt_g[bm].mean()) - float(yp_g[bm].mean()))
                ece += (cnt / n_g) * gap
        g_val = g.item() if hasattr(g, "item") else g
        raw_eces[str(g_val)] = ece
        rows.append({
            "group": g_val,
            "n": n_g,
            "ece": round(ece, 4),
            "brier": round(float(brier_score_loss(yt_g, yp_g)), 4),
        })
    worst = max(raw_eces, key=raw_eces.get)
    max_gap = round(max(raw_eces.values()) - min(raw_eces.values()), 4)
    return {
        "per_group": pd.DataFrame(
            rows, columns=["group", "n", "ece", "brier"]
        ),
        "worst_ece_group": worst,
        "max_ece_gap": max_gap,
    }

cal_wealth = subgroup_calibration_audit(
    y_te, p_te, df_te["wealth_quintile"], n_bins=5
)
print(cal_wealth["per_group"].to_string(index=False))
print(
    f"Worst ECE group: {cal_wealth['worst_ece_group']} | "
    f"Max ECE gap: {cal_wealth['max_ece_gap']:.4f}"
)


## Section 34.3.4 — Qualitative Error Analysis: Worst False Negatives & False Positives

In [ ]:
def slice_worst_errors(df, y_true, y_prob, top_k=5):
    """Return top-k most confident false negatives and false positives."""
    work = df.copy()
    work["y_true"] = np.asarray(y_true, dtype=int)
    work["y_prob"] = np.asarray(y_prob, dtype=float)
    fn_df = (
        work[work["y_true"] == 1]
        .sort_values("y_prob", ascending=True)
        .head(top_k)
        .reset_index(drop=True)
    )
    fp_df = (
        work[work["y_true"] == 0]
        .sort_values("y_prob", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    return {
        "worst_false_negatives": fn_df,
        "worst_false_positives": fp_df,
    }

view_cols = ["household_id", "residence", "wealth_quintile", "anc_visits"]
errs = slice_worst_errors(df_te[view_cols], y_te, np.round(p_te, 4), 3)
print("--- Top 3 Missed Stunting Cases (Worst FN) ---")
print(errs["worst_false_negatives"].to_string(index=False))
print("\n--- Top 3 False Alarms (Worst FP) ---")
print(errs["worst_false_positives"].to_string(index=False))


## Section 34.3.5 — Post-Processing Mitigation Without Retraining

In [ ]:
def calibrate_group_thresholds_for_tpr(
    y_true, y_prob, groups, target_tpr=0.75
):
    """Pick highest per-group cutoff achieving TPR >= target_tpr."""
    if not (0.0 < target_tpr <= 1.0):
        raise ValueError("target_tpr must lie in (0, 1]")
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_prob, dtype=float)
    gr = np.asarray(groups)
    out = {}
    for g in sorted(pd.unique(gr)):
        m = gr == g
        yt_g, yp_g = yt[m], yp[m]
        n_pos = int(np.sum(yt_g == 1))
        best_t = 0.0
        if n_pos > 0:
            for t in np.sort(np.unique(yp_g))[::-1]:
                if np.sum((yt_g == 1) & (yp_g >= t)) / n_pos >= target_tpr:
                    best_t = float(t)
                    break
        out[g.item() if hasattr(g, "item") else g] = round(best_t, 4)
    return out

# Leak-free Train / Val / Test split on maternal_anemia (no Hb feature!)
anem_cols = ["maternal_age_years", "anc_visits", "wealth_rank", "is_urban"]
anem_df = df[["residence", "maternal_anemia"] + anem_cols].dropna()
Xa, ya, ga = (
    anem_df[anem_cols],
    anem_df["maternal_anemia"].astype(int),
    anem_df["residence"],
)
Xa_tr, Xa_tmp, ya_tr, ya_tmp, ga_tr, ga_tmp = train_test_split(
    Xa, ya, ga, test_size=0.50, stratify=ya, random_state=42
)
Xa_val, Xa_te, ya_val, ya_te, ga_val, ga_te = train_test_split(
    Xa_tmp, ya_tmp, ga_tmp, test_size=0.50, stratify=ya_tmp, random_state=42
)
anem_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500)),
])
anem_pipe.fit(Xa_tr, ya_tr)
pa_val = anem_pipe.predict_proba(Xa_val)[:, 1]
pa_te = anem_pipe.predict_proba(Xa_te)[:, 1]

before_gaps = compute_fairness_gaps(
    ya_te, (pa_te >= 0.50).astype(int), ga_te
)
grp_th = calibrate_group_thresholds_for_tpr(
    ya_val, pa_val, ga_val, target_tpr=0.75
)
pred_after = np.array(
    [int(p >= grp_th[g]) for p, g in zip(pa_te, ga_te)], dtype=int
)
after_gaps = compute_fairness_gaps(ya_te, pred_after, ga_te)

print("Validation-calibrated thresholds:", grp_th)
print(
    f"Equal Opportunity gap: {before_gaps['equal_opportunity_diff']:.4f} "
    f"-> {after_gaps['equal_opportunity_diff']:.4f}"
)


## Section 34.3.6 — Covariate Shift Monitoring with Population Stability Index (PSI)

In [ ]:
def population_stability_index(expected, actual, n_bins=10, eps=1e-6):
    """Compute PSI and drift severity status from quantile bins."""
    exp = np.asarray(expected, dtype=float)
    act = np.asarray(actual, dtype=float)
    if len(exp) == 0 or len(act) == 0:
        raise ValueError("expected and actual arrays must be non-empty")
    edges = np.unique(np.quantile(exp, np.linspace(0.0, 1.0, n_bins + 1)))
    if len(edges) < 2:
        edges = np.array([-np.inf, np.inf])
    else:
        edges = edges.copy()
        edges[0] = -np.inf
        edges[-1] = np.inf
    exp_cnt, _ = np.histogram(exp, bins=edges)
    act_cnt, _ = np.histogram(act, bins=edges)
    p_exp = np.clip(exp_cnt / exp_cnt.sum(), eps, None)
    p_exp = p_exp / p_exp.sum()
    p_act = np.clip(act_cnt / act_cnt.sum(), eps, None)
    p_act = p_act / p_act.sum()
    psi = round(float(np.sum((p_act - p_exp) * np.log(p_act / p_exp))), 4)
    if psi < 0.10:
        status = "Stable"
    elif psi < 0.25:
        status = "Moderate Drift"
    else:
        status = "Severe Drift"
    return {"psi": psi, "drift_status": status}

# Simulate pandemic/famine shock: clinic visits and maternal Hb drop
rng = np.random.default_rng(42)
shock_df = clean.copy()
shock_df["maternal_age_years"] = (
    shock_df["maternal_age_years"] + rng.normal(0.0, 0.4, len(shock_df))
)
shock_df["anc_visits"] = np.clip(shock_df["anc_visits"] - 1.5, 0.0, None)
shock_df["maternal_hb_g_dl"] = shock_df["maternal_hb_g_dl"] - 1.2

for col in ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]:
    res = population_stability_index(clean[col], shock_df[col])
    print(f"{col:20s} | PSI={res['psi']:.4f} | {res['drift_status']}")


## Section 34.3.7 — Break It On Purpose! (Responsible AI Audit Error Anatomy)

In [ ]:
# Intentional Bug #1: Mismatched length between y_true and sensitive_attr
compute_fairness_gaps([0, 1, 1, 0], [0, 1, 0, 0], ["Rural", "Urban"])


In [ ]:
# Intentional Bug #2: Empty baseline array passed to PSI quantile binning
population_stability_index([], [10.5, 11.2, 12.1])


In [ ]:
# Intentional Bug #3: Impossible target_tpr=1.5 outside (0, 1]
calibrate_group_thresholds_for_tpr(
    [0, 1, 0, 1], [0.2, 0.8, 0.3, 0.7],
    ["Rural", "Rural", "Urban", "Urban"], target_tpr=1.5,
)


## Section 34.4 — Stage 4 Blueprint: End-to-End Responsible Clinical AI Audit Gate

In [ ]:
def build_responsible_ai_audit(raw_df, current_df, seed=42):
    """Run fairness, calibration, and shift deployment gate."""
    work = raw_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["anc_visits"] = work["anc_visits"].replace(999, np.nan)
    work["is_urban"] = (work["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    work["wealth_rank"] = work["wealth_quintile"].map(w_map)
    fcols = [
        "maternal_age_years", "anc_visits", "maternal_hb_g_dl",
        "wealth_rank", "is_urban",
    ]
    sub = work[["child_stunted", "residence", "wealth_quintile"] + fcols]
    sub = sub.dropna().reset_index(drop=True)
    X_tr, X_te, y_tr, y_te, d_tr, d_te = train_test_split(
        sub[fcols], sub["child_stunted"].astype(int), sub,
        test_size=0.30, stratify=sub["child_stunted"], random_state=seed,
    )
    mdl = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500)),
    ])
    mdl.fit(X_tr, y_tr)
    fpr_t, tpr_t, th_t = roc_curve(y_tr, mdl.predict_proba(X_tr)[:, 1])
    cut = float(th_t[int(np.argmax(tpr_t - fpr_t))])
    pr_te = mdl.predict_proba(X_te)[:, 1]
    fg = compute_fairness_gaps(
        y_te, (pr_te >= cut).astype(int), d_te["residence"]
    )
    cal = subgroup_calibration_audit(
        y_te, pr_te, d_te["wealth_quintile"], n_bins=5
    )
    drift_cols = ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]
    max_psi = max(
        population_stability_index(sub[c], current_df[c].dropna())["psi"]
        for c in drift_cols
    )
    gate = (
        "PASS"
        if (fg["equal_opportunity_diff"] < 0.15 and max_psi < 0.25)
        else "BLOCK — Audit Required"
    )
    return {
        "overall_auc": round(float(roc_auc_score(y_te, pr_te)), 4),
        "youden_t": round(cut, 4),
        "eo_gap": fg["equal_opportunity_diff"],
        "worst_ece_group": cal["worst_ece_group"],
        "max_ece_gap": cal["max_ece_gap"],
        "max_psi": round(max_psi, 4),
        "deployment_gate": gate,
    }

audit = build_responsible_ai_audit(mch_raw, shock_df)
for k, v in audit.items():
    print(f"{k:16s}: {v}")
